# GNN parameter-matched runs: Transformer (seven feature configurations)

The fixed architecture (`conf/hyperparameters/gnn/fixed.yaml`) with one change: `hidden` is 236 instead of 128, so that Transformer has about as many parameters as PNA at hidden 128. Everything else is the same as in `gnn_fixed_architecture_runs.ipynb`.

| Config | Transformer params, hidden 236 | PNA params, hidden 128 | Difference |
|---|---|---|---|
| c1 | 565,693 | 561,537 | +0.7% |
| c2 | 575,133 | 599,681 | -4.1% |
| c3 | 580,089 | 569,345 | +1.9% |
| c4 | 618,321 | 623,105 | -0.8% |
| c5 | 589,529 | 607,489 | -3.0% |
| c6 | 584,809 | 604,929 | -3.3% |
| c7 | 575,369 | 566,785 | +1.5% |

Hidden 236 is the multiple of 4 with the smallest worst difference over the seven configurations (4.1%).

- Settings (right panel): Accelerator = GPU, Internet = On.
- Input: the dataset `sandrokhizanishvili/hi-small-gnn` (graphs + feature_meta.json).
- The runs stop after 11 h: Kaggle ends a session at 12 h and keeps no output of a session it ends. The finished runs are saved; run the missing configurations in a second session (`FEATURE_CONFIGS`).
- Each run writes `results.json`, `history.csv`, `curves.png`, `predictions.csv`, `best.pt`, `config.yaml` to `/kaggle/working/outputs/train/transformer/<run>/seed-42/`. The run names are the ones of the fixed-architecture runs: see "After the run" before copying anything.

In [ ]:
MODEL = 'transformer'
HIDDEN = 236             # 128 in the fixed architecture
FEATURE_CONFIGS = 'c1,c2,c3,c4,c5,c6,c7'
SEED = 42

In [ ]:
# The pinned torch / PyG stack known to work on Kaggle GPU; do not change versions.
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch_geometric lightgbm==4.6.0 hydra-core==1.3.2

In [ ]:
# The code: clone once, pull on a rerun.
import os
if os.path.isdir('/kaggle/working/gnn_for_aml'):
    !cd /kaggle/working/gnn_for_aml && git pull
else:
    !git clone https://github.com/sandrokhizanishvili/gnn_for_aml.git /kaggle/working/gnn_for_aml
%cd /kaggle/working/gnn_for_aml
!ls /kaggle/input/datasets/sandrokhizanishvili/hi-small-gnn

In [ ]:
# Environment test: versions, the GPU, one forward / backward pass per operator.
!python run.py mode=check machine=kaggle

In [ ]:
# The runs, one after another, stopped after 11 h (the session limit is 12 h). A finished run is skipped on a rerun.
!timeout 11h python run.py -m model={MODEL} feature_config={FEATURE_CONFIGS} seed={SEED} hyperparameters.hidden={HIDDEN} machine=kaggle

In [ ]:
# The metrics of every finished run, one row each, with the parameter count next to PNA's and the time per epoch.
import glob, json, pandas as pd
PNA_PARAMS = {('none', 'base'): 561537, ('base', 'base'): 599681, ('none', 'full'): 569345, ('full', 'full'): 623105,
              ('base', 'full'): 607489, ('base', 'gfp'): 604929, ('none', 'gfp'): 566785}   # PNA at hidden 128, by (mp, readout)
rows = []
for path in sorted(glob.glob(f'/kaggle/working/outputs/train/{MODEL}/*/seed-*/results.json')):
    row = json.load(open(path))
    assert row['recipe']['hidden'] == HIDDEN, f"{row['run']}: hidden {row['recipe']['hidden']}, expected {HIDDEN}"
    row['pna_params'] = PNA_PARAMS[row['mp'], row['readout']]
    row['params_vs_pna_pct'] = 100 * (row['params'] - row['pna_params']) / row['pna_params']
    row['seconds_per_epoch'] = pd.read_csv(path.replace('results.json', 'history.csv'))['seconds'].mean()
    rows.append(row)
cols = ['run', 'params', 'pna_params', 'params_vs_pna_pct', 'seconds_per_epoch', 'best_epoch', 'threshold'] + [f'{s}_{m}' for s in ('train', 'val', 'test') for m in ('f1', 'precision', 'recall', 'pr_auc')]
pd.DataFrame(rows)[cols].round(4) if rows else "no finished run yet"

In [ ]:
# A small archive for download: the finished runs without the checkpoints and predictions, with _h-<HIDDEN> added
# to every run folder, so that unzipping it into outputs/ cannot touch the fixed-architecture runs of the same name.
import os, zipfile
with zipfile.ZipFile('/kaggle/working/results_small.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for done in sorted(glob.glob(f'/kaggle/working/outputs/train/{MODEL}/*/seed-*/results.json')):
        run, seed = done.split('/')[-3:-1]
        for name in sorted(os.listdir(os.path.dirname(done))):
            if not name.endswith('.pt') and name != 'predictions.csv':
                archive.write(f'{os.path.dirname(done)}/{name}', f'outputs/train/{MODEL}/{run}_h-{HIDDEN}/{seed}/{name}')
!ls -lh /kaggle/working/results_small.zip

## After the run
- Output tab: download `results_small.zip` (metrics, curves, histories). Its run folders already end in `_h-236`: unzip it into `gnn_for_aml/outputs/` on the laptop.
- The whole `outputs` folder of the Output tab (adds `predictions.csv` and `best.pt`) keeps the fixed-architecture run names. Add `_h-236` to every run folder before copying it into `outputs/train/transformer/`, or it overwrites the hidden-128 runs.
- The `run` field inside `results.json` keeps the name without the suffix; `recipe.hidden` and `params` tell the two sets of runs apart.
- Commit the small files (`predictions.csv` and `*.pt` are gitignored); mark the rows in `EXPERIMENTS.md`.